# ARGUS Text Risk Demo

Enter raw text, choose a retriever and thresholds, then run named-entity extraction plus ARGUS risk scoring. The first run may download the NER or embedding model.

## Setup

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "with_argus_eyes").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not find the With_Argus_Eyes repository root from this notebook location.")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = lambda value: value
    display = print

from with_argus_eyes.inference import (
    ArgusTextConfig,
    analyze_text,
    available_retrievers,
    highlight_entities,
    resolve_model_artifact,
)

print("Available retrievers:", ", ".join(available_retrievers()))

## User configuration

In [ ]:
config = ArgusTextConfig(
    retriever="contriever",
    language="en",
    ner_model="dslim/bert-base-NER",
    risk_threshold=0.3,
    ner_threshold=0.5,
    order=800,
    k=50,
    text_mode="context",
    workspace_root=repo_root,
)

artifact = resolve_model_artifact(config)
print("Selected ARGUS model artifact:")
print(artifact)

## Text input

In [ ]:
text = """
St. Martin's Church in Zillis, Switzerland, is a Romanesque church best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.
""".strip()

print(text)

## Run analysis

In [ ]:
results = analyze_text(text, config)

if not results:
    print("No named entities were found with the current NER threshold.")
else:
    print(f"Scored {len(results)} entity mentions.")

## Results table

In [ ]:
columns = ["entity", "entity_type", "ner_score", "risk_score", "above_threshold", "retriever"]

if results:
    try:
        import pandas as pd
        display(pd.DataFrame(results)[columns].sort_values("risk_score", ascending=False))
    except ImportError:
        for row in sorted(results, key=lambda item: item["risk_score"], reverse=True):
            print({key: row[key] for key in columns})
else:
    print("Nothing to display.")

## Highlighted text

In [ ]:
if results:
    display(HTML("<div style='line-height:1.8; font-size:1rem'>" + highlight_entities(text, results) + "</div>"))
else:
    print("No highlighted entities.")